In [ ]:
from idlelib.outwin import file_line_helper
from pickle import FALSE

import numpy as np
import rioxarray 
import rasterio as rio
from rasterio.plot import show
from rasterio.enums import Resampling
import glob
import os
from osgeo import gdal
from pathlib import Path
from scipy.ndimage import median_filter

#from win32comext.axscript.client.framework import profile



In [ ]:
LiDAR_dir = "C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR"
raster_files = glob.glob(os.path.join(LiDAR_dir, '*.tif'))

### To begin, I will simply re-sample our 0.5-m rasters. I am preserving this code so I have an example of the GDAL structure

In [ ]:
##Using GDAL
extent = (601558.000, 4862467.500, 609431.500, 4870872.500)

for raster in raster_files:
    out_dir = "C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR/resampled"
    name, ext = os.path.splitext(os.path.basename(raster))
    out_raster = os.path.join(out_dir, f"{name}_100{ext}")
    gdal.Warp(
        out_raster, 
        raster, 
        dstNodata=float("nan"),
        xRes=100,
        yRes=100,
        resampleAlg="average",
        outputBounds=extent)

### The same thins with Rasterio, again so that I have the structure


In [ ]:
#Using Rasterio
for raster in raster_files:
    dir_name = os.path.dirname(raster)
    name, ext = os.path.splitext(os.path.basename(raster))
    out_raster = os.path.join(dir_name, f"{name}_100_rasterio{ext}")
    sd = rioxarray.open_rasterio(raster)
    sd.rio.write_nodata(np.nan, inplace=True)  # make sure nans are counted as no data
    sd_downsampled = sd.rio.reproject(
        sd.rio.crs,
        resolution = (100,100),
        resampling=Resampling.average)
    sd_downsampled.rio.to_raster(out_raster)


## We dedcided to try and rasterize our point cloud data directly. 
### Resampled the following output: ~/ice-road/results/pc-transform/laz-snow-trans_source.laz
used following pipeline on Borah:\
        [ \
    { \
        "type": "readers.las", \
        "filename": "$laz"\
    }, \
    { \
        "type": "writers.gdal", \
        "filename": "$tif", \
        "resolution": 100, \
        "output_type": "mean", \
        "dimension": "Z", \
        "data_type": "float32", \
        "nodata": -9999 \
    } \
] \
Outputs saved to C:\Users\RDCRLSMC\Desktop\SIRO\LiDAR2

In [ ]:
# set up directory paths
# I will need to difference the re-rasterized point clouds from the reference, which also needs to be resampled

dir = Path("C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR2")
tifs = glob.glob(os.path.join(dir, "*.tif"))
ref = "C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR2/ref/MCS_REFDEM_WGS84_cubic.tif"

In [ ]:
#I need to see if raster profiles match
with rio.open(ref) as ref_src:
    ref_src.read(1, masked=True)
    print(ref_src.profile)
    
for raster in tifs:
    with rio.open(raster) as src:
        src.read(1, masked=True)
        print(src.profile)

In [ ]:
# Profiles do not match, so I need to resample
# Need to pull grid info from ref

with rio.open(ref) as ref_src:
    ref_dem = ref_src.read(1, masked=True)
    ref_profile = ref_src.profile
    ref_width = ref_src.width
    ref_height = ref_src.height
    ref_transform = ref_src.transform

#resample rasters prior to differencing 

for raster_file in tifs:
    with rio.open(raster_file) as src:
        # Resample to match reference dimensions
        data = src.read(1,
            out_shape=(ref_height, ref_width),
            resampling=Resampling.bilinear  # DEMs usually bilinear
        )

        # Update profile for new raster
        out_profile = ref_profile
        out_profile.update({
            "dtype": data.dtype,
            "height": ref_height,
            "width": ref_width,
            "transform": ref_transform,
            "nodata": -9999
        })

        # Write aligned raster
        resampled_dir = os.path.join(dir, "resampled")
        os.makedirs(resampled_dir, exist_ok=True)  # creates it if it doesn't exist
        out_path = os.path.join(resampled_dir, os.path.basename(raster_file))
        with rio.open(out_path, "w", **out_profile) as dst:
            dst.write(data, 1)

print("All rasters aligned to reference raster.")

In [ ]:
## Difference rasters

for raster in glob.glob(os.path.join(resampled_dir, "*.tif")): 
    with rio.open(raster) as src:
        base = os.path.basename(raster).split("-")[0]   
        out_raster = os.path.join(dir, f"{base}_sd.tif")
        
        # Read DTM and metadata
        dtm = src.read(1, masked=True)
        dsm_meta = src.profile

        # Compute snow depth
        sd = dtm - ref_dem

        # Write output raster
        with rio.open(out_raster, "w", **dsm_meta) as dst:
            dst.write(sd.astype("float32"), 1)

### Some of these rasters deviate from the reference raster by over 100 (close to 200) m. Therefore, returning to manipulation of 0.5-m rasters
### I think there is an issue with averaging the point cloud over a 100-m region when there are data gaps 

In [55]:
#chose a single test dataset

LiDAR_dir = Path("C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR")

raster_files = glob.glob(os.path.join(LiDAR_dir, '*.tif'))


#raster_files = glob.glob(os.path.join(LiDAR_dir, "SNEX_MCS_Lidar_20250404_SD_V01.0.tif"))

for raster in raster_files:
    print(raster)
    #with rio.open(raster) as src:
        #sd = src.read(1, masked=True)
        #print(sd.max(), sd.min())
        #print(src.shape)
      

C:\Users\RDCRLSMC\Desktop\SIRO\LiDAR\SNEX_MCS_Lidar_20221208_SD_V01.0.tif
C:\Users\RDCRLSMC\Desktop\SIRO\LiDAR\SNEX_MCS_Lidar_20230209_SD_V01.0.tif
C:\Users\RDCRLSMC\Desktop\SIRO\LiDAR\SNEX_MCS_Lidar_20230316_SD_V01.0.tif
C:\Users\RDCRLSMC\Desktop\SIRO\LiDAR\SNEX_MCS_Lidar_20230405_SD_V01.0.tif
C:\Users\RDCRLSMC\Desktop\SIRO\LiDAR\SNEX_MCS_Lidar_20231228_SD_V01.0.tif
C:\Users\RDCRLSMC\Desktop\SIRO\LiDAR\SNEX_MCS_Lidar_20240115_SD_V01.0.tif
C:\Users\RDCRLSMC\Desktop\SIRO\LiDAR\SNEX_MCS_Lidar_20240213_SD_V01.0.tif
C:\Users\RDCRLSMC\Desktop\SIRO\LiDAR\SNEX_MCS_Lidar_20240315_SD_V01.0.tif
C:\Users\RDCRLSMC\Desktop\SIRO\LiDAR\SNEX_MCS_Lidar_20240418_SD_V01.0.tif
C:\Users\RDCRLSMC\Desktop\SIRO\LiDAR\SNEX_MCS_Lidar_20250113_SD_V01.0.tif
C:\Users\RDCRLSMC\Desktop\SIRO\LiDAR\SNEX_MCS_Lidar_20250129_SD_V01.0.tif
C:\Users\RDCRLSMC\Desktop\SIRO\LiDAR\SNEX_MCS_Lidar_20250404_SD_V01.0.tif
C:\Users\RDCRLSMC\Desktop\SIRO\LiDAR\SNEX_MCS_Lidar_20250501_SD_V01.0.tif


In [ ]:
# Remove below 0 values and remove highest 1% of values (outliers)
# I found that most negative values are along the road or in areas of no snow or what I believe to be melt
# I want to see though if the +/- 3 standard deviations from the mean accounts for these values
#doesn't work- a lot of values are way beyond three SD at highest elevations

# for raster in raster_files:
#     with rio.open(raster) as src:
#         sd = src.read(1, masked=True).filled(np.nan)
#         mean_val = np.nanmean(sd)
#         std_dev  = np.nanstd(sd)
#         threshold = 3 * std_dev
#         print(raster)
#         print( {mean_val},{std_dev},{mean_val + threshold},{mean_val - threshold} )

In [56]:
# https://docs.scipy.org/doc/scipy/tutorial/ndimage.html#filter-functions

for raster in raster_files:
    with rio.open(raster) as src:
        out_raster = os.path.join(LiDAR_dir / "filtered", os.path.basename(raster).replace(".tif", "_filtered.tif"))
        #print(out_raster)
        #print(src.shape)
        sd = src.read(1, masked=True).filled(np.nan)  # <-- fill masked with NaN
        sd[sd < 0] = np.nan
        sd[sd > 10] = np.nan
        meta = src.profile
        # Write output raster
        with rio.open(out_raster, "w", **meta) as dst:
            dst.write(sd.astype("float32"), 1)
        
        

In [ ]:
        sd = median_filter(sd, size=10, mode='nearest')
             # size is number of pixels, so 20 = 10-m
        #vals = sd[np.isfinite(sd)]
        #upper = np.percentile(vals, 99.9)
        #sd[sd > upper] = np.nan
        #print(np.nanmax(sd), np.nanmin(sd))
        meta = src.profile
        # Write output raster
        with rio.open(out_raster, "w", **meta) as dst:
            dst.write(sd.astype("float32"), 1)


In [64]:
import rasterio
from rasterio.fill import fillnodata

#creating huge holes
# need to return to this
# https://rasterio.readthedocs.io/en/stable/api/rasterio.fill.html#rasterio.fill.fillnodata

raster = Path("C:/Users/RDCRLSMC/Desktop/SIRO/LiDAR/filtered/SNEX_MCS_Lidar_20250501_SD_V01.0_filtered.tif")
dir_name = os.path.dirname(raster)
name, ext = os.path.splitext(os.path.basename(raster))
out_raster = os.path.join(dir_name, f"{name}_fill-blah{ext}")
sd = rioxarray.open_rasterio(raster)

#sd.rio.write_nodata(np.nan, inplace=True)  # make sure nans are counted as no data
# 1. Open the elevation file
with rasterio.open(raster) as src:
    profile = src.profile
    # Read the first band and the mask (identifies NoData areas)
    image = src.read(1, masked=True)

# 2. Fill the gaps
# max_search_distance: max pixels to search for valid values
# smoothing_iterations: number of passes to smooth filled areas
filled_data = fillnodata(image, max_search_distance=2, smoothing_iterations=0)

In [65]:

# 3. Write the output
with rasterio.open(out_raster, 'w', **profile) as dst:
    dst.write(filled_data, 1)


In [ ]:
#Resample rasters to 100-m

upscale_factor = 200

for raster in glob.glob(os.path.join(LiDAR_dir / "filtered", "*.tif")):
    dir_name = os.path.dirname(raster)
    name, ext = os.path.splitext(os.path.basename(raster))
    out_raster = os.path.join(dir_name, f"{name}_100_rasterio{ext}")
    sd = rioxarray.open_rasterio(raster)
    sd.rio.write_nodata(np.nan, inplace=True)  # make sure nans are counted as no data
    sd_downsampled = sd.rio.reproject(
        sd.rio.crs,
        resolution = (100,100),
        resampling=Resampling.average)
    sd_downsampled.rio.to_raster(out_raster)        